In [ ]:
from typing import Self

import pydantic

In [ ]:
import marimo as mo

In [ ]:
import os
from pathlib import Path
from pprint import pprint
import datetime
import random
import dotenv
import pandas
import plotly.express
import altair
from ortools.sat.python import cp_model
import amplify_sched
import didppy

# ジョブショップスケジューリング問題

$J || C_{\max}$ と書く.

- ジョブ $J_1, \dots, J_n$
- ジョブ $J_j$ に属するオペレーション $O_{1j}, \dots, O_{m_jj}$. この順で処理される.
- 機械 $M_1, \dots, M_m$
- オペレーション $O_{ij}$ は機械 $\mu_{ij}$ で作業時間 $p_{ij}$ かけて処理する.
- オペレーションは中断できない
- 最後のオペレーションの終了時刻を最小化

In [ ]:
parent = str(Path(os.path.abspath(__file__)).parent)
data_dir = os.path.join(parent, "data")

In [ ]:
class Task(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(frozen=True)

    machine: int = pydantic.Field(..., ge=0, frozen=True)
    time: int = pydantic.Field(..., ge=0, frozen=True)

In [ ]:
class Job(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(frozen=True)

    tasks: list[Task] = pydantic.Field(frozen=True)

    @classmethod
    def from_file(cls, fname: str) -> list[Self]:
        with open(fname) as f:
            n, m = None, None
            machine, proc_time = {}, {}

            i = 0
            for line in f:
                if line[0] == "#":
                    continue

                if n is None or m is None:
                    n, m = map(int, line.split())
                    print(f"{n=}, {m=}")
                    continue

                L = list(map(int, line.split()))
                for j in range(m):
                    machine[i, j] = L[2 * j]
                    proc_time[i, j] = L[2 * j + 1]
                i += 1

        jobs = []
        for i in range(n):
            tasks = []
            for j in range(m):
                tasks.append(Task(machine=machine[i, j], time=proc_time[i, j]))
            jobs.append(cls(tasks=tasks))

        return jobs

In [ ]:
def plot_plotly(df: pandas.DataFrame):
    return plotly.express.timeline(
        df,
        x_start="start",
        x_end="end",
        y="resource",
        color="job",
        opacity=0.5,
    ).update_yaxes(categoryorder="category descending")

In [ ]:
def plot_altair(df: pandas.DataFrame):
    return (
        altair.Chart(df)
        .mark_bar()
        .encode(
            x="start",
            x2="end",
            y="resource",
            color="job",
        )
        .properties(width="container", height=400)
    )

In [ ]:
fname1 = os.path.join(data_dir, "ft06.txt")
jobs1 = Job.from_file(fname1)
pprint(jobs1)

n=6, m=6
[Job(tasks=[Task(machine=2, time=1), Task(machine=0, time=3), Task(machine=1, time=6), Task(machine=3, time=7), Task(machine=5, time=3), Task(machine=4, time=6)]),
 Job(tasks=[Task(machine=1, time=8), Task(machine=2, time=5), Task(machine=4, time=10), Task(machine=5, time=10), Task(machine=0, time=10), Task(machine=3, time=4)]),
 Job(tasks=[Task(machine=2, time=5), Task(machine=3, time=4), Task(machine=5, time=8), Task(machine=0, time=9), Task(machine=1, time=1), Task(machine=4, time=7)]),
 Job(tasks=[Task(machine=1, time=5), Task(machine=0, time=5), Task(machine=2, time=5), Task(machine=3, time=3), Task(machine=4, time=8), Task(machine=5, time=9)]),
 Job(tasks=[Task(machine=2, time=9), Task(machine=1, time=3), Task(machine=4, time=5), Task(machine=5, time=4), Task(machine=0, time=3), Task(machine=3, time=1)]),
 Job(tasks=[Task(machine=1, time=3), Task(machine=3, time=3), Task(machine=5, time=9), Task(machine=0, time=10), Task(machine=4, time=4), Task(machine=2, time=1)])]


## OR-Tools による求解

In [ ]:
class ModelCpSat:
    def __init__(self, jobs: list[Job]):
        self.jobs = jobs
        self.model = cp_model.CpModel()
        num_machines = len(
            set(task.machine for job in self.jobs for task in job.tasks)
        )
        self.machines = list(range(num_machines))
        horizon = sum(task.time for job in self.jobs for task in job.tasks)

        self.starts = [[None for task in job.tasks] for job in jobs]
        self.intervals = [[None for task in job.tasks] for job in jobs]
        machine_to_interval = {m: [] for m in self.machines}

        for id_job, job in enumerate(self.jobs):
            for id_task, task in enumerate(job.tasks):
                suffix = f"_{id_job}_{id_task}"
                start = self.model.new_int_var(0, horizon, "start" + suffix)
                interval = self.model.new_fixed_size_interval_var(
                    start, task.time, "interval" + suffix
                )
                self.starts[id_job][id_task] = start
                self.intervals[id_job][id_task] = interval
                machine_to_interval[task.machine].append(interval)

        for machine in machine_to_interval:
            if len(machine_to_interval[machine]) > 0:
                self.model.add_no_overlap(machine_to_interval[machine])

        for id_job, job in enumerate(self.jobs):
            for id_task, task in enumerate(job.tasks):
                if id_task > 0:
                    curr = self.intervals[id_job][id_task]
                    prev = self.intervals[id_job][id_task - 1]
                    self.model.add(curr.start_expr() >= prev.end_expr())

        makespan = self.model.new_int_var(0, horizon, "makespan")
        self.model.add_max_equality(
            makespan,
            [
                self.intervals[id_job][-1].end_expr()
                for id_job, job in enumerate(self.jobs)
            ],
        )
        self.model.minimize(makespan)

    def solve(self, timeout: int = 10):
        self.solver = cp_model.CpSolver()
        self.solver.parameters.log_search_progress = True
        self.solver.parameters.max_time_in_seconds = timeout
        self.status = self.solver.solve(self.model)

    def to_df(self) -> pandas.DataFrame:
        today = datetime.date.today()
        l = []
        for id_job, job in enumerate(self.jobs):
            for id_task, task in enumerate(job.tasks):
                start = self.solver.value(
                    self.intervals[id_job][id_task].start_expr()
                )
                end = start + self.jobs[id_job].tasks[id_task].time
                l.append(
                    dict(
                        job=f"job{id_job}",
                        task=f"task{id_task}",
                        resource=f"machine{self.jobs[id_job].tasks[id_task].machine}",
                        start=today + datetime.timedelta(start),
                        end=today + datetime.timedelta(end),
                    )
                )
        df = pandas.DataFrame(l)
        df["start"] = pandas.to_datetime(df["start"])
        df["end"] = pandas.to_datetime(df["end"])
        return df

In [ ]:
model1_cpsat = ModelCpSat(jobs1)
model1_cpsat.solve()


Starting CP-SAT solver v9.15.6755
Parameters: max_time_in_seconds: 10 log_search_progress: true
Setting number of workers to 12

Initial optimization model '': (model_fingerprint: 0x43f0af3eddfba1f7)
#Variables: 37 (#ints: 1 in objective) (36 primary variables)
  - 37 in [0,197]
#kInterval: 36
#kLinMax: 1 (#expressions: 6)
#kLinear2: 30
#kNoOverlap: 6 (#intervals: 36)

Starting presolve at 0.00s
  1.47e-05s  0.00e+00d  [DetectDominanceRelations] 
  3.18e-04s  0.00e+00d  [PresolveToFixPoint] #num_loops=7 #num_dual_strengthening=1 
  1.12e-06s  0.00e+00d  [ExtractEncodingFromLinear] 
  3.94e-06s  0.00e+00d  [DetectDuplicateColumns] 
  1.60e-05s  0.00e+00d  [DetectDuplicateConstraints] 
[Symmetry] Graph for symmetry has 146 nodes and 175 arcs.
[Symmetry] Symmetry computation done. time: 1.8866e-05 dtime: 1.547e-05
  1.43e-05s  0.00e+00d  [DetectDuplicateConstraintsWithDifferentEnforcements] 
  1.64e-04s  7.42e-07d  [Probe] 
  1.82e-06s  0.00e+00d  [MaxClique] 
  1.66e-05s  0.00e+00d  [De

In [ ]:
plot_plotly(model1_cpsat.to_df())

<marimo-plotly data-figure='{"data":[{"base":["2026-09-28T00:00:00","2026-09-29T00:00:00","2026-10-09T00:00:00","2026-10-23T00:00:00","2026-10-31T00:00:00","2026-11-04T00:00:00"],"hovertemplate":"job=job0\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job0","marker":{"color":"#636efa","opacity":0.5,"pattern":{"shape":""}},"name":"job0","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"AFwmBQAUcw8AKOYeAIQMJAAUcw8AKOYe"},"xaxis":"x","y":["machine2","machine0","machine1","machine3","machine5","machine4"],"yaxis":"y","type":"bar"},{"base":["2026-09-23T00:00:00","2026-10-01T00:00:00","2026-10-06T00:00:00","2026-10-21T00:00:00","2026-10-31T00:00:00","2026-11-10T00:00:00"],"hovertemplate":"job=job1\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job1","marker":{"color":"#EF553B","opacity":0.5,"pattern":{"shape":""}},"name":"job1","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"AOAyKQDMvxkAmH8zAJh/MwCYfzMAcJkU"},"xaxis":"x","y":["machine1","machine2","machine4","machine5","machine0","machine3"],"yaxis":"y","type":"bar"},{"base":["2026-09-23T00:00:00","2026-09-28T00:00:00","2026-10-02T00:00:00","2026-10-11T00:00:00","2026-10-20T00:00:00","2026-11-10T00:00:00"],"hovertemplate":"job=job2\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job2","marker":{"color":"#00cc96","opacity":0.5,"pattern":{"shape":""}},"name":"job2","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"AMy/GQBwmRQA4DIpADxZLgBcJgUAhAwk"},"xaxis":"x","y":["machine2","machine3","machine5","machine0","machine1","machine4"],"yaxis":"y","type":"bar"},{"base":["2026-10-01T00:00:00","2026-10-06T00:00:00","2026-10-15T00:00:00","2026-10-20T00:00:00","2026-10-23T00:00:00","2026-11-07T00:00:00"],"hovertemplate":"job=job3\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job3","marker":{"color":"#ab63fa","opacity":0.5,"pattern":{"shape":""}},"name":"job3","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"AMy/GQDMvxkAzL8ZABRzDwDgMikAPFku"},"xaxis":"x","y":["machine1","machine0","machine2","machine3","machine4","machine5"],"yaxis":"y","type":"bar"},{"base":["2026-10-06T00:00:00","2026-10-15T00:00:00","2026-10-18T00:00:00","2026-11-03T00:00:00","2026-11-10T00:00:00","2026-11-14T00:00:00"],"hovertemplate":"job=job4\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job4","marker":{"color":"#FFA15A","opacity":0.5,"pattern":{"shape":""}},"name":"job4","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"ADxZLgAUcw8AzL8ZAHCZFAAUcw8AXCYF"},"xaxis":"x","y":["machine2","machine1","machine4","machine5","machine0","machine3"],"yaxis":"y","type":"bar"},{"base":["2026-10-06T00:00:00","2026-10-09T00:00:00","2026-10-12T00:00:00","2026-10-21T00:00:00","2026-10-31T00:00:00","2026-11-04T00:00:00"],"hovertemplate":"job=job5\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job5","marker":{"color":"#19d3f3","opacity":0.5,"pattern":{"shape":""}},"name":"job5","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"ABRzDwAUcw8APFkuAJh/MwBwmRQAXCYF"},"xaxis":"x","y":["machine1","machine3","machine5","machine0","machine4","machine2"],"yaxis":"y","type":"bar"}],"layout":{"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,

In [ ]:
mo.ui.altair_chart(plot_altair(model1_cpsat.to_df()))

In [ ]:
plot_altair(model1_cpsat.to_df())

## 数理最適化ソルバーによる求解

HiGHS で離接定式化を解くのは `jobshop_highs.py` に分離.

OR-Tools と highspy はどっちも HiGHS を `libhighs.so.1` という同じ名前で同梱していて,
同じプロセスで両方 import すると後から読んだ方がバージョン違いの HiGHS を掴んで `ImportError` になる.
(ortools 9.15.6755 の中身は HiGHS 1.12.0, highspy 1.15.1 は HiGHS 1.15.1)

## FIXSTARS Amplify Scheduling Engine による求解

In [ ]:
dotenv.load_dotenv(dotenv.find_dotenv(usecwd=True))
token = os.environ["FIXSTARS_SE"]


class ModelAmplifySe:
    def __init__(self, jobs: list[Job]):
        self.jobs = jobs
        num_machines = len(
            set(task.machine for job in self.jobs for task in job.tasks)
        )
        self.machines = list(range(num_machines))
        self.se_machines = [
            amplify_sched.Machine(name=f"machine{midx}")
            for midx in self.machines
        ]

        self.model = amplify_sched.Model()

        for semachine in self.se_machines:
            self.model.machines.add(machine=semachine)

        self.se_jobs = [
            amplify_sched.Job(name=f"job{jidx}")
            for jidx, _ in enumerate(self.jobs)
        ]
        for idx, job in enumerate(self.jobs):
            sejob = self.se_jobs[idx]
            self.model.jobs.add(sejob)
            for jdx, task in enumerate(job.tasks):
                semachine = self.se_machines[task.machine]
                setask = amplify_sched.Task()
                setask.processing_times[semachine] = task.time
                self.model.jobs[sejob.name].append(setask)

    def solve(self, timeout: int = 5) -> None:
        self.solution = self.model.solve(token=token, timeout=timeout)

    def get_makespan(self) -> int:
        return int(self.solution.table["Finish"].max())

    def to_df(self) -> pandas.DataFrame:
        sol_df = self.solution.table
        today = datetime.date.today()
        l = []
        for id_job, job in enumerate(self.jobs):
            sejob = self.se_jobs[id_job]
            for id_task, task in enumerate(job.tasks):
                start = int(
                    sol_df[sol_df["Job"] == sejob.name]["Start"].reset_index(
                        drop=True
                    )[id_task]
                )
                end = start + self.jobs[id_job].tasks[id_task].time
                l.append(
                    dict(
                        job=f"job{id_job}",
                        task=f"task{id_task}",
                        resource=f"machine{self.jobs[id_job].tasks[id_task].machine}",
                        start=today + datetime.timedelta(start),
                        end=today + datetime.timedelta(end),
                    )
                )
        df = pandas.DataFrame(l)
        df["start"] = pandas.to_datetime(df["start"])
        df["end"] = pandas.to_datetime(df["end"])
        return df

In [ ]:
model1_amplify = ModelAmplifySe(jobs1)
model1_amplify.solve()

print(f"makespan = {model1_amplify.get_makespan()}")

makespan = 55


In [ ]:
model1_amplify.solution.timeline(machine_view=True)

<marimo-plotly data-figure='{"data":[{"base":[5,6,16,22,42,49],"customdata":[["job0",0],["job0",1],["job0",2],["job0",3],["job0",4],["job0",5]],"hovertemplate":"Job=%{customdata[0]}\u003cbr\u003eStart=%{base}\u003cbr\u003eFinish=%{x}\u003cbr\u003eMachine=%{y}\u003cbr\u003eProcess=%{customdata[1]}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job0","marker":{"color":"#636efa","pattern":{"shape":""}},"name":"job0","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i1","bdata":"AQMGBwMG"},"xaxis":"x","y":["machine2","machine0","machine1","machine3","machine5","machine4"],"yaxis":"y","type":"bar"},{"base":[0,8,13,28,38,48],"customdata":[["job1",0],["job1",1],["job1",2],["job1",3],["job1",4],["job1",5]],"hovertemplate":"Job=%{customdata[0]}\u003cbr\u003eStart=%{base}\u003cbr\u003eFinish=%{x}\u003cbr\u003eMachine=%{y}\u003cbr\u003eProcess=%{customdata[1]}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job1","marker":{"color":"#EF553B","pattern":{"shape":""}},"name":"job1","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i1","bdata":"CAUKCgoE"},"xaxis":"x","y":["machine1","machine2","machine4","machine5","machine0","machine3"],"yaxis":"y","type":"bar"},{"base":[0,5,9,18,27,30],"customdata":[["job2",0],["job2",1],["job2",2],["job2",3],["job2",4],["job2",5]],"hovertemplate":"Job=%{customdata[0]}\u003cbr\u003eStart=%{base}\u003cbr\u003eFinish=%{x}\u003cbr\u003eMachine=%{y}\u003cbr\u003eProcess=%{customdata[1]}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job2","marker":{"color":"#00cc96","pattern":{"shape":""}},"name":"job2","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i1","bdata":"BQQICQEH"},"xaxis":"x","y":["machine2","machine3","machine5","machine0","machine1","machine4"],"yaxis":"y","type":"bar"},{"base":[8,13,22,29,37,45],"customdata":[["job3",0],["job3",1],["job3",2],["job3",3],["job3",4],["job3",5]],"hovertemplate":"Job=%{customdata[0]}\u003cbr\u003eStart=%{base}\u003cbr\u003eFinish=%{x}\u003cbr\u003eMachine=%{y}\u003cbr\u003eProcess=%{customdata[1]}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job3","marker":{"color":"#ab63fa","pattern":{"shape":""}},"name":"job3","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i1","bdata":"BQUFAwgJ"},"xaxis":"x","y":["machine1","machine0","machine2","machine3","machine4","machine5"],"yaxis":"y","type":"bar"},{"base":[13,22,25,38,48,52],"customdata":[["job4",0],["job4",1],["job4",2],["job4",3],["job4",4],["job4",5]],"hovertemplate":"Job=%{customdata[0]}\u003cbr\u003eStart=%{base}\u003cbr\u003eFinish=%{x}\u003cbr\u003eMachine=%{y}\u003cbr\u003eProcess=%{customdata[1]}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job4","marker":{"color":"#FFA15A","pattern":{"shape":""}},"name":"job4","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i1","bdata":"CQMFBAMB"},"xaxis":"x","y":["machine2","machine1","machine4","machine5","machine0","machine3"],"yaxis":"y","type":"bar"},{"base":[13,16,19,28,45,49],"customdata":[["job5",0],["job5",1],["job5",2],["job5",3],["job5",4],["job5",5]],"hovertemplate":"Job=%{customdata[0]}\u003cbr\u003eStart=%{base}\u003cbr\u003eFinish=%{x}\u003cbr\u003eMachine=%{y}\u003cbr\u003eProcess=%{customdata[1]}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job5","marker":{"color":"#19d3f3","pattern":{"shape":""}},"name":"job5","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i1","bdata":"AwMJCgQB"},"xaxis":"x","y":["machine1","machine3","machine5","machine0","machine4","machine2"],"yaxis":"y","type":"bar"}],"layout":{"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}]

In [ ]:
plot_altair(model1_amplify.to_df())

## より大きい問題

In [ ]:
def gen_data(n_jobs: int, n_machines: int) -> list[Job]:
    random.seed(0)
    jobs = []
    for id_job in range(n_jobs):
        machines = list(range(n_machines))
        random.shuffle(machines)
        tasks = []
        for id_task in range(n_machines):
            machine = machines[id_task]
            time = random.randint(1, 10)
            tasks.append(Task(machine=machine, time=time))

        jobs.append(Job(tasks=tasks))

    return jobs

In [ ]:
jobs2 = gen_data(45, 15)
pprint(jobs2)

[Job(tasks=[Task(machine=1, time=9), Task(machine=10, time=3), Task(machine=9, time=5), Task(machine=5, time=3), Task(machine=11, time=2), Task(machine=2, time=10), Task(machine=3, time=5), Task(machine=7, time=9), Task(machine=8, time=10), Task(machine=4, time=3), Task(machine=0, time=5), Task(machine=14, time=2), Task(machine=12, time=2), Task(machine=6, time=6), Task(machine=13, time=8)]),
 Job(tasks=[Task(machine=13, time=2), Task(machine=11, time=7), Task(machine=10, time=1), Task(machine=0, time=10), Task(machine=2, time=8), Task(machine=4, time=6), Task(machine=14, time=4), Task(machine=7, time=6), Task(machine=3, time=2), Task(machine=9, time=4), Task(machine=12, time=10), Task(machine=6, time=4), Task(machine=5, time=4), Task(machine=1, time=3), Task(machine=8, time=9)]),
 Job(tasks=[Task(machine=10, time=10), Task(machine=6, time=9), Task(machine=3, time=10), Task(machine=11, time=5), Task(machine=0, time=8), Task(machine=2, time=2), Task(machine=9, time=10), Task(machine=4, 

In [ ]:
model2_cpsat = ModelCpSat(jobs2)
model2_cpsat.solve()


Starting CP-SAT solver v9.15.6755
Parameters: max_time_in_seconds: 10 log_search_progress: true
Setting number of workers to 12

Initial optimization model '': (model_fingerprint: 0x62b4a1d48ce14446)
#Variables: 676 (#ints: 1 in objective) (675 primary variables)
  - 676 in [0,3683]
#kInterval: 675
#kLinMax: 1 (#expressions: 45)
#kLinear2: 630
#kNoOverlap: 15 (#intervals: 675)

Starting presolve at 0.00s
  1.22e-04s  0.00e+00d  [DetectDominanceRelations] 
  9.34e-03s  0.00e+00d  [PresolveToFixPoint] #num_loops=16 #num_dual_strengthening=1 
  4.16e-06s  0.00e+00d  [ExtractEncodingFromLinear] 
  1.87e-05s  0.00e+00d  [DetectDuplicateColumns] 
  1.45e-04s  0.00e+00d  [DetectDuplicateConstraints] 
[Symmetry] Graph for symmetry has 2'672 nodes and 3'331 arcs.
[Symmetry] Symmetry computation done. time: 0.000293316 dtime: 0.00031934
  1.46e-04s  0.00e+00d  [DetectDuplicateConstraintsWithDifferentEnforcements] 
  1.20e-03s  3.92e-06d  [Probe] #new_bounds=1 
  3.16e-06s  0.00e+00d  [MaxClique

In [ ]:
plot_plotly(model2_cpsat.to_df())

<marimo-plotly data-figure='{"data":[{"base":["2026-10-02T00:00:00","2026-10-20T00:00:00","2026-11-02T00:00:00","2026-11-11T00:00:00","2027-01-14T00:00:00","2027-01-19T00:00:00","2027-02-18T00:00:00","2027-02-23T00:00:00","2027-03-10T00:00:00","2027-04-06T00:00:00","2027-05-08T00:00:00","2027-05-18T00:00:00","2027-05-20T00:00:00","2027-05-24T00:00:00","2027-06-03T00:00:00"],"hovertemplate":"job=job0\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job0","marker":{"color":"#636efa","opacity":0.5,"pattern":{"shape":""}},"name":"job0","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"ADxZLgAUcw8AzL8ZABRzDwC4TAoAmH8zAMy/GQA8WS4AmH8zABRzDwDMvxkAuEwKALhMCgAo5h4A4DIp"},"xaxis":"x","y":["machine1","machine10","machine9","machine5","machine11","machine2","machine3","machine7","machine8","machine4","machine0","machine14","machine12","machine6","machine13"],"yaxis":"y","type":"bar"},{"base":["2026-09-25T00:00:00","2026-10-18T00:00:00","2026-10-29T00:00:00","2026-10-30T00:00:00","2026-11-22T00:00:00","2026-12-27T00:00:00","2027-01-11T00:00:00","2027-03-13T00:00:00","2027-03-29T00:00:00","2027-04-01T00:00:00","2027-04-10T00:00:00","2027-04-20T00:00:00","2027-04-24T00:00:00","2027-05-12T00:00:00","2027-05-15T00:00:00"],"hovertemplate":"job=job1\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job1","marker":{"color":"#EF553B","opacity":0.5,"pattern":{"shape":""}},"name":"job1","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"ALhMCgCEDCQAXCYFAJh/MwDgMikAKOYeAHCZFAAo5h4AuEwKAHCZFACYfzMAcJkUAHCZFAAUcw8APFku"},"xaxis":"x","y":["machine13","machine11","machine10","machine0","machine2","machine4","machine14","machine7","machine3","machine9","machine12","machine6","machine5","machine1","machine8"],"yaxis":"y","type":"bar"},{"base":["2026-09-23T00:00:00","2026-10-07T00:00:00","2026-12-03T00:00:00","2026-12-23T00:00:00","2026-12-28T00:00:00","2027-01-09T00:00:00","2027-01-19T00:00:00","2027-02-03T00:00:00","2027-03-01T00:00:00","2027-03-25T00:00:00","2027-04-27T00:00:00","2027-05-10T00:00:00","2027-05-15T00:00:00","2027-05-21T00:00:00","2027-06-06T00:00:00"],"hovertemplate":"job=job2\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job2","marker":{"color":"#00cc96","opacity":0.5,"pattern":{"shape":""}},"name":"job2","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"AJh/MwA8WS4AmH8zAMy/GQDgMikAuEwKAJh/MwCEDCQAKOYeAJh/MwBwmRQAzL8ZABRzDwBwmRQAFHMP"},"xaxis":"x","y":["machine10","machine6","machine3","machine11","machine0","machine2","machine9","machine4","machine12","machine14","machine8","machine5","machine13","machine1","machine7"],"yaxis":"y","type":"bar"},{"base":["2026-09-24T00:00:00","2026-10-03T00:00:00","2026-10-12T00:00:00","2026-10-16T00:00:00","2026-10-26T00:00:00","2026-11-02T00:00:00","2026-12-12T00:00:00","2027-01-01T00:00:00","2027-01-09T00:00:00","2027-01-25T00:00:00","2027-03-04T00:00:00","2027-04-09T00:00:00","2027-06-25T00:00:00","2027-07-05T00:00:00","2027-07-07T00:00:00"],"hovertemplate":"job=job3\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job3","marker":{"color":"#ab63fa","opacity":0.5,"pattern":{"shape":""}},"name":"job3","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"ADxZLgBwmRQAcJkUAJh/MwCEDCQAmH8zAMy/GQDgMikA4DIpACjmHgC4TAoAKOYeAJh/MwC4TAoA4DIp"},"xaxis":"x","y":["machine11","machine6","machine12","machine3","machine5","machine14","machine8","machine2","machine13","machine1","machine7","machine4","machine10","machine9","machine0"],"yaxis":"y","type":"bar"},{"base":["2026-09-23T00:00:00","2026-10-10T00:00:00","2026-10-17T00:00:00","2026-10-29T00:00:00","2026-11-02T00:00:00",

In [ ]:
model2_amplify = ModelAmplifySe(jobs2)
model2_amplify.solve(timeout=5)

print(f"makespan = {model2_amplify.get_makespan()}")

makespan = 297


In [ ]:
plot_plotly(model2_amplify.to_df())

<marimo-plotly data-figure='{"data":[{"base":["2026-10-09T00:00:00","2026-11-04T00:00:00","2026-11-09T00:00:00","2026-11-16T00:00:00","2026-12-17T00:00:00","2026-12-23T00:00:00","2027-01-22T00:00:00","2027-01-28T00:00:00","2027-03-20T00:00:00","2027-04-04T00:00:00","2027-04-07T00:00:00","2027-05-16T00:00:00","2027-05-18T00:00:00","2027-05-20T00:00:00","2027-06-20T00:00:00"],"hovertemplate":"job=job0\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job0","marker":{"color":"#636efa","opacity":0.5,"pattern":{"shape":""}},"name":"job0","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"ADxZLgAUcw8AzL8ZABRzDwC4TAoAmH8zAMy/GQA8WS4AmH8zABRzDwDMvxkAuEwKALhMCgAo5h4A4DIp"},"xaxis":"x","y":["machine1","machine10","machine9","machine5","machine11","machine2","machine3","machine7","machine8","machine4","machine0","machine14","machine12","machine6","machine13"],"yaxis":"y","type":"bar"},{"base":["2026-10-03T00:00:00","2026-10-18T00:00:00","2026-11-19T00:00:00","2026-11-20T00:00:00","2026-11-30T00:00:00","2026-12-27T00:00:00","2027-01-04T00:00:00","2027-01-08T00:00:00","2027-03-29T00:00:00","2027-04-01T00:00:00","2027-04-06T00:00:00","2027-04-21T00:00:00","2027-05-30T00:00:00","2027-06-03T00:00:00","2027-06-06T00:00:00"],"hovertemplate":"job=job1\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job1","marker":{"color":"#EF553B","opacity":0.5,"pattern":{"shape":""}},"name":"job1","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"ALhMCgCEDCQAXCYFAJh/MwDgMikAKOYeAHCZFAAo5h4AuEwKAHCZFACYfzMAcJkUAHCZFAAUcw8APFku"},"xaxis":"x","y":["machine13","machine11","machine10","machine0","machine2","machine4","machine14","machine7","machine3","machine9","machine12","machine6","machine5","machine1","machine8"],"yaxis":"y","type":"bar"},{"base":["2026-09-23T00:00:00","2026-10-12T00:00:00","2026-10-28T00:00:00","2026-11-26T00:00:00","2026-12-07T00:00:00","2027-01-08T00:00:00","2027-01-18T00:00:00","2027-02-08T00:00:00","2027-03-08T00:00:00","2027-03-14T00:00:00","2027-04-29T00:00:00","2027-06-03T00:00:00","2027-06-17T00:00:00","2027-06-26T00:00:00","2027-07-13T00:00:00"],"hovertemplate":"job=job2\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job2","marker":{"color":"#00cc96","opacity":0.5,"pattern":{"shape":""}},"name":"job2","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"AJh/MwA8WS4AmH8zAMy/GQDgMikAuEwKAJh/MwCEDCQAKOYeAJh/MwBwmRQAzL8ZABRzDwBwmRQAFHMP"},"xaxis":"x","y":["machine10","machine6","machine3","machine11","machine0","machine2","machine9","machine4","machine12","machine14","machine8","machine5","machine13","machine1","machine7"],"yaxis":"y","type":"bar"},{"base":["2026-09-23T00:00:00","2026-10-08T00:00:00","2026-10-13T00:00:00","2026-10-18T00:00:00","2026-10-28T00:00:00","2026-11-07T00:00:00","2027-01-11T00:00:00","2027-02-04T00:00:00","2027-02-12T00:00:00","2027-03-03T00:00:00","2027-03-28T00:00:00","2027-04-13T00:00:00","2027-04-19T00:00:00","2027-05-19T00:00:00","2027-07-04T00:00:00"],"hovertemplate":"job=job3\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job3","marker":{"color":"#ab63fa","opacity":0.5,"pattern":{"shape":""}},"name":"job3","orientation":"h","showlegend":true,"textposition":"auto","x":{"dtype":"i4","bdata":"ADxZLgBwmRQAcJkUAJh/MwCEDCQAmH8zAMy/GQDgMikA4DIpACjmHgC4TAoAKOYeAJh/MwC4TAoA4DIp"},"xaxis":"x","y":["machine11","machine6","machine12","machine3","machine5","machine14","machine8","machine2","machine13","machine1","machine7","machine4","machine10","machine9","machine0"],"yaxis":"y","type":"bar"},{"base":["2026-10-12T00:00:00","2026-10-19T00:00:00","2026-10-29T00:00:00","2026-11-04T00:00:00","2026-11-08T00:00:00",

## 他のインスタンス

ta50 は最適解は知られていない.

bounds

- upper: 1923
- lower: 1833

In [ ]:
instance_dir = os.path.join(parent, "jsplib/instances")

In [ ]:
fname3 = os.path.join(instance_dir, "ta50")
jobs3 = Job.from_file(fname3)

n=30, m=20


In [ ]:
model3_amplify = ModelAmplifySe(jobs3)
model3_amplify.solve(timeout=10)

mo.md(f"makespan = {model3_amplify.get_makespan()}")

makespan = 2090

In [ ]:
plot_plotly(model3_amplify.to_df())

<marimo-plotly data-figure='{"data":[{"base":["2026-12-22T00:00:00","2027-06-18T00:00:00","2027-07-14T00:00:00","2027-08-17T00:00:00","2027-10-06T00:00:00","2027-11-18T00:00:00","2027-12-20T00:00:00","2028-03-07T00:00:00","2028-07-23T00:00:00","2029-01-20T00:00:00","2029-05-14T00:00:00","2029-08-05T00:00:00","2029-10-10T00:00:00","2029-12-23T00:00:00","2030-04-08T00:00:00","2030-07-02T00:00:00","2030-09-12T00:00:00","2031-01-31T00:00:00","2032-01-28T00:00:00","2032-03-09T00:00:00"],"hovertemplate":"job=job0\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job0","marker":{"color":"#636efa","opacity":0.5,"pattern":{"shape":""}},"name":"job0","orientation":"h","showlegend":true,"textposition":"auto","x":[8294400000,2246400000,2851200000,1641600000,3715200000,1468800000,2246400000,5702400000,7257600000,4838400000,7171200000,5702400000,6393600000,2073600000,7344000000,4060800000,7603200000,8380800000,3542400000,6652800000],"xaxis":"x","y":["machine2","machine8","machine6","machine0","machine4","machine15","machine16","machine13","machine5","machine12","machine10","machine7","machine14","machine19","machine1","machine3","machine18","machine17","machine9","machine11"],"yaxis":"y","type":"bar"},{"base":["2026-09-23T00:00:00","2027-01-30T00:00:00","2027-04-12T00:00:00","2027-07-30T00:00:00","2027-10-14T00:00:00","2028-01-21T00:00:00","2028-03-24T00:00:00","2028-11-10T00:00:00","2028-12-14T00:00:00","2029-02-18T00:00:00","2029-05-08T00:00:00","2029-08-13T00:00:00","2030-02-07T00:00:00","2030-04-14T00:00:00","2030-06-05T00:00:00","2030-09-22T00:00:00","2030-11-23T00:00:00","2031-05-03T00:00:00","2031-12-14T00:00:00","2032-03-17T00:00:00"],"hovertemplate":"job=job1\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job1","marker":{"color":"#EF553B","opacity":0.5,"pattern":{"shape":""}},"name":"job1","orientation":"h","showlegend":true,"textposition":"auto","x":[6048000000,3974400000,7776000000,5270400000,2073600000,5443200000,8208000000,2937600000,4060800000,4320000000,5356800000,864000000,5702400000,4492800000,4233600000,345600000,8121600000,3283200000,8035200000,7257600000],"xaxis":"x","y":["machine0","machine10","machine13","machine3","machine2","machine7","machine1","machine16","machine9","machine17","machine18","machine15","machine11","machine8","machine19","machine5","machine4","machine12","machine14","machine6"],"yaxis":"y","type":"bar"},{"base":["2027-03-08T00:00:00","2027-06-18T00:00:00","2027-09-22T00:00:00","2028-04-09T00:00:00","2028-08-13T00:00:00","2029-09-11T00:00:00","2030-04-15T00:00:00","2030-06-10T00:00:00","2030-07-12T00:00:00","2030-08-17T00:00:00","2030-09-22T00:00:00","2030-12-05T00:00:00","2031-03-18T00:00:00","2031-06-17T00:00:00","2031-07-16T00:00:00","2031-09-24T00:00:00","2031-11-13T00:00:00","2031-12-04T00:00:00","2032-01-09T00:00:00","2032-02-14T00:00:00"],"hovertemplate":"job=job2\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job2","marker":{"color":"#00cc96","opacity":0.5,"pattern":{"shape":""}},"name":"job2","orientation":"h","showlegend":true,"textposition":"auto","x":[4147200000,5184000000,1296000000,2160000000,1814400000,8553600000,4838400000,2764800000,2678400000,3110400000,6393600000,6220800000,7862400000,2505600000,2937600000,4320000000,1814400000,3110400000,86400000,2592000000],"xaxis":"x","y":["machine0","machine19","machine11","machine9","machine15","machine3","machine13","machine18","machine1","machine5","machine7","machine6","machine10","machine4","machine14","machine8","machine12","machine2","machine16","machine17"],"yaxis":"y","type":"bar"},{"base":["2026-10-06T00:00:00","2026-11-25T00:00:00","2027-04-26T00:00:00","2027-10-07T00:00:00","2027-11-12T00:00:00","2028-02-24T00:00:00","2028-04-18T00:00:00","2028-05-22T00:00:00","2028-08-18T00:00:00","2028-11-08T00:00:0

In [ ]:
model3_cpsat = ModelCpSat(jobs3)
model3_cpsat.solve(timeout=10)

mo.md(f"makespan = {round(model3_cpsat.solver.objective_value)}")


Starting CP-SAT solver v9.15.6755
Parameters: max_time_in_seconds: 10 log_search_progress: true
Setting number of workers to 12

Initial optimization model '': (model_fingerprint: 0x33f82f20921558d4)
#Variables: 601 (#ints: 1 in objective) (600 primary variables)
  - 601 in [0,30657]
#kInterval: 600
#kLinMax: 1 (#expressions: 30)
#kLinear2: 570
#kNoOverlap: 20 (#intervals: 600)

Starting presolve at 0.00s
  1.01e-04s  0.00e+00d  [DetectDominanceRelations] 
  1.05e-02s  0.00e+00d  [PresolveToFixPoint] #num_loops=21 #num_dual_strengthening=1 
  4.19e-06s  0.00e+00d  [ExtractEncodingFromLinear] 
  1.90e-05s  0.00e+00d  [DetectDuplicateColumns] 
  1.38e-04s  0.00e+00d  [DetectDuplicateConstraints] 
[Symmetry] Graph for symmetry has 2'392 nodes and 2'971 arcs.
[Symmetry] Symmetry computation done. time: 0.000127332 dtime: 0.0002559
  1.35e-04s  0.00e+00d  [DetectDuplicateConstraintsWithDifferentEnforcements] 
  9.80e-04s  4.02e-06d  [Probe] #new_bounds=1 
  3.14e-06s  0.00e+00d  [MaxClique

makespan = 2122

In [ ]:
plot_plotly(model3_cpsat.to_df())

<marimo-plotly data-figure='{"data":[{"base":["2027-04-02T00:00:00","2027-07-07T00:00:00","2027-08-02T00:00:00","2027-09-04T00:00:00","2027-10-01T00:00:00","2027-11-13T00:00:00","2028-01-17T00:00:00","2028-03-07T00:00:00","2028-05-13T00:00:00","2028-09-28T00:00:00","2028-12-07T00:00:00","2029-02-28T00:00:00","2029-05-05T00:00:00","2029-07-18T00:00:00","2029-08-26T00:00:00","2029-11-19T00:00:00","2030-03-19T00:00:00","2030-06-24T00:00:00","2031-04-25T00:00:00","2031-10-16T00:00:00"],"hovertemplate":"job=job0\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job0","marker":{"color":"#636efa","opacity":0.5,"pattern":{"shape":""}},"name":"job0","orientation":"h","showlegend":true,"textposition":"auto","x":[8294400000,2246400000,2851200000,1641600000,3715200000,1468800000,2246400000,5702400000,7257600000,4838400000,7171200000,5702400000,6393600000,2073600000,7344000000,4060800000,7603200000,8380800000,3542400000,6652800000],"xaxis":"x","y":["machine2","machine8","machine6","machine0","machine4","machine15","machine16","machine13","machine5","machine12","machine10","machine7","machine14","machine19","machine1","machine3","machine18","machine17","machine9","machine11"],"yaxis":"y","type":"bar"},{"base":["2026-09-23T00:00:00","2027-01-30T00:00:00","2027-03-23T00:00:00","2027-06-30T00:00:00","2027-10-14T00:00:00","2027-12-17T00:00:00","2028-02-18T00:00:00","2028-05-29T00:00:00","2028-09-17T00:00:00","2028-11-03T00:00:00","2028-12-25T00:00:00","2029-02-25T00:00:00","2030-02-22T00:00:00","2030-04-29T00:00:00","2030-06-20T00:00:00","2030-08-08T00:00:00","2030-08-12T00:00:00","2031-12-12T00:00:00","2032-01-19T00:00:00","2032-04-21T00:00:00"],"hovertemplate":"job=job1\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job1","marker":{"color":"#EF553B","opacity":0.5,"pattern":{"shape":""}},"name":"job1","orientation":"h","showlegend":true,"textposition":"auto","x":[6048000000,3974400000,7776000000,5270400000,2073600000,5443200000,8208000000,2937600000,4060800000,4320000000,5356800000,864000000,5702400000,4492800000,4233600000,345600000,8121600000,3283200000,8035200000,7257600000],"xaxis":"x","y":["machine0","machine10","machine13","machine3","machine2","machine7","machine1","machine16","machine9","machine17","machine18","machine15","machine11","machine8","machine19","machine5","machine4","machine12","machine14","machine6"],"yaxis":"y","type":"bar"},{"base":["2027-04-05T00:00:00","2027-07-20T00:00:00","2027-09-18T00:00:00","2027-10-03T00:00:00","2027-12-10T00:00:00","2028-01-18T00:00:00","2028-05-12T00:00:00","2028-07-07T00:00:00","2028-09-11T00:00:00","2029-05-06T00:00:00","2029-11-10T00:00:00","2030-09-15T00:00:00","2030-12-16T00:00:00","2031-04-04T00:00:00","2031-05-03T00:00:00","2031-06-25T00:00:00","2031-09-08T00:00:00","2031-09-29T00:00:00","2031-11-04T00:00:00","2031-11-08T00:00:00"],"hovertemplate":"job=job2\u003cbr\u003estart=%{base}\u003cbr\u003eend=%{x}\u003cbr\u003eresource=%{y}\u003cextra\u003e\u003c/extra\u003e","legendgroup":"job2","marker":{"color":"#00cc96","opacity":0.5,"pattern":{"shape":""}},"name":"job2","orientation":"h","showlegend":true,"textposition":"auto","x":[4147200000,5184000000,1296000000,2160000000,1814400000,8553600000,4838400000,2764800000,2678400000,3110400000,6393600000,6220800000,7862400000,2505600000,2937600000,4320000000,1814400000,3110400000,86400000,2592000000],"xaxis":"x","y":["machine0","machine19","machine11","machine9","machine15","machine3","machine13","machine18","machine1","machine5","machine7","machine6","machine10","machine4","machine14","machine8","machine12","machine2","machine16","machine17"],"yaxis":"y","type":"bar"},{"base":["2026-09-23T00:00:00","2026-11-13T00:00:00","2027-07-24T00:00:00","2027-10-06T00:00:00","2028-01-13T00:00:00","2028-05-07T00:00:00","2028-07-20T00:00:00","2029-03-02T00:00:00","2029-06-24T00:00:00","2029-09-14T00:00:0

## didp での求解

### 状態

- $\text{Q}$: set 変数. 配置されていないタスクの集合を表す.
- $\text{tm}_m \space (\forall m: \text{machine})$: 機械ごとに makespan を保持する.
- $\text{tj}_j \space (\forall j: \text{job})$: ジョブごとに makespan を保持する.

### 目的関数

- $\text{makespan} := \max \{ \text{tm}_m, \text{tj}_j \mid m: \text{machine}, \space j: \text{job} \}$

### 更新規則

- タスク $\text{task} \in Q$ は全ての先行タスクが $Q$ に属していない時配置可能.
- $\text{task}$ が配置された場合, それを $Q$ から取り除く.
- $\text{task}$ が配置された場合, タスクを処理する機械 $m$ とタスクの属するジョブ $j$ に対して以下のように更新する.
    - $\text{tm}_m \leftarrow \max(\text{tm}_m + t_\text{task}, \space \text{tj}_j + t_\text{task})$
    - $\text{tj}_j \leftarrow \max(\text{tm}_m + t_\text{task}, \space \text{tj}_j + t_\text{task})$
- 上記更新の後, 目的関数を再計算する.

In [ ]:
class ModelDidp:
    def __init__(self, jobs: list[Job]):
        self.jobs = jobs
        n_tasks = sum(len(job.tasks) for job in self.jobs)
        n_machines = len(
            set(task.machine for job in self.jobs for task in job.tasks)
        )

        self.model = didppy.Model()

        objtype_task = self.model.add_object_type(number=n_tasks)

        remaining = self.model.add_set_var(
            object_type=objtype_task, target=list(range(n_tasks))
        )

        cur_time_per_machine = [
            # self.model.add_int_var(target=0) for _ in range(n_machines)
            self.model.add_int_resource_var(target=0, less_is_better=True)
            for _ in range(n_machines)
        ]
        cur_time_per_job = [
            # self.model.add_int_var(target=0) for _ in self.jobs
            self.model.add_int_resource_var(target=0, less_is_better=True)
            for _ in self.jobs
        ]

        self.model.add_base_case([remaining.is_empty()])

        # task_to_time = self.model.add_int_table(
        #     [task.time for job in self.jobs for task in job.tasks]
        # )
        # task_to_machine = self.model.add_int_table(
        #     [task.machine for job in self.jobs for task in job.tasks]
        # )

        precs = []
        id_jobtask = 0
        for id_job, job in enumerate(self.jobs):
            prec = set()
            for id_task, task in enumerate(job.tasks):
                precs.append(prec.copy())
                prec.add(id_jobtask)
                id_jobtask += 1

        task_to_prec = self.model.add_set_table(
            precs, object_type=objtype_task
        )

        id_jobtask = 0
        for id_job, job in enumerate(self.jobs):
            for id_task, task in enumerate(job.tasks):
                sched = didppy.Transition(
                    name=f"sched_job{id_job}_task{id_task}",
                    cost=(
                        didppy.max(
                            didppy.IntExpr.state_cost(),
                            didppy.max(
                                cur_time_per_machine[task.machine] + task.time,
                                cur_time_per_job[id_job] + task.time,
                            ),
                        )
                    ),
                    effects=[
                        (remaining, remaining.remove(id_jobtask)),
                        (
                            cur_time_per_job[id_job],
                            didppy.max(
                                cur_time_per_machine[task.machine] + task.time,
                                cur_time_per_job[id_job] + task.time,
                            ),
                        ),
                        (
                            cur_time_per_machine[task.machine],
                            didppy.max(
                                cur_time_per_machine[task.machine] + task.time,
                                cur_time_per_job[id_job] + task.time,
                            ),
                        ),
                    ],
                    preconditions=[
                        remaining.contains(id_jobtask),
                        remaining.isdisjoint(task_to_prec[id_jobtask]),
                    ],
                )
                self.model.add_transition(sched)

                id_jobtask += 1

        task_to_min_cost = []
        for job in self.jobs:
            cost = sum(task.time for task in job.tasks)
            for task in job.tasks:
                task_to_min_cost.append(cost)
                cost -= task.time
        task_to_min_cost_table = self.model.add_int_table(task_to_min_cost)
        self.model.add_dual_bound(
            remaining.is_empty().if_then_else(
                0, task_to_min_cost_table.min(remaining)
            )
        )

    def solve(self, timeout=10, threads: int = 8) -> None:
        self.solver = didppy.CABS(
            self.model, threads=threads, quiet=False, time_limit=timeout
        )
        # self.solver = didppy.LNBS(
        #     self.model, threads=threads, quiet=False, time_limit=timeout
        # )
        self.solution: didppy.Solution = self.solver.search()

In [ ]:
model1_didp = ModelDidp(jobs1)
model1_didp.solve(threads=10, timeout=10)

mo.md(f"makespan = {round(model1_didp.solution.cost)}")

Solver: CABS from DIDPPy v0.11.1
Searched with beam size: 1, threads: 10, kept: 174, sent: 0
Searched with beam size: 1, expanded: 36, elapsed time: 0.000278599
New primal bound: 84, expanded: 36, elapsed time: 0.000281274
Searched with beam size: 2, threads: 10, kept: 149, sent: 194
Searched with beam size: 2, expanded: 104, elapsed time: 0.000558129
New dual bound: 4, expanded: 104, elapsed time: 0.000559361
New primal bound: 77, expanded: 104, elapsed time: 0.000572747
Searched with beam size: 4, threads: 10, kept: 143, sent: 501
Searched with beam size: 4, expanded: 230, elapsed time: 0.000905207
Searched with beam size: 8, threads: 10, kept: 183, sent: 1146
Searched with beam size: 8, expanded: 483, elapsed time: 0.001601418
New primal bound: 71, expanded: 483, elapsed time: 0.001615886
Searched with beam size: 16, threads: 10, kept: 272, sent: 2323
Searched with beam size: 16, expanded: 978, elapsed time: 0.002402368
Searched with beam size: 32, threads: 10, kept: 515, sent: 4702

makespan = 55

In [ ]:
model1_didp.solution.is_optimal

False

In [ ]:
for _t in model1_didp.solution.transitions:
    print(_t.name)

sched_job1_task0
sched_job2_task0
sched_job0_task0
sched_job1_task1
sched_job4_task0
sched_job3_task0
sched_job0_task1
sched_job1_task2
sched_job2_task1
sched_job3_task1
sched_job5_task0
sched_job2_task2
sched_job0_task2
sched_job3_task2
sched_job4_task1
sched_job5_task1
sched_job0_task3
sched_job5_task2
sched_job2_task3
sched_job3_task3
sched_job2_task4
sched_job4_task2
sched_job1_task3
sched_job5_task3
sched_job4_task3
sched_job2_task5
sched_job3_task4
sched_job0_task4
sched_job1_task4
sched_job4_task4
sched_job1_task5
sched_job5_task4
sched_job0_task5
sched_job3_task5
sched_job4_task5
sched_job5_task5


In [ ]:
model2_didp = ModelDidp(jobs2)
model2_didp.solve(threads=10, timeout=10)

mo.md(f"makespan = {round(model2_didp.solution.cost)}")

Solver: CABS from DIDPPy v0.11.1
Searched with beam size: 1, threads: 10, kept: 23775, sent: 0
Searched with beam size: 1, expanded: 675, elapsed time: 0.035342318
New primal bound: 370, expanded: 675, elapsed time: 0.035344953
Searched with beam size: 2, threads: 10, kept: 23419, sent: 23233
Searched with beam size: 2, expanded: 2024, elapsed time: 0.076483991
New dual bound: 2, expanded: 2024, elapsed time: 0.076486375
New primal bound: 345, expanded: 2024, elapsed time: 0.077017884
Searched with beam size: 4, threads: 10, kept: 23736, sent: 70759
Searched with beam size: 4, expanded: 4685, elapsed time: 0.12605102
Searched with beam size: 8, threads: 10, kept: 23581, sent: 166089
Searched with beam size: 8, expanded: 10028, elapsed time: 0.210035172
Searched with beam size: 16, threads: 10, kept: 36980, sent: 330445
Searched with beam size: 16, expanded: 20641, elapsed time: 0.359710047
Searched with beam size: 32, threads: 10, kept: 74574, sent: 674713
Searched with beam size: 32, 

makespan = 330

In [ ]:
model3_didp = ModelDidp(jobs3)
model3_didp.solve(threads=10, timeout=10)

mo.md(f"makespan = {round(model3_didp.solution.cost)}")

Solver: CABS from DIDPPy v0.11.1
Searched with beam size: 1, threads: 10, kept: 15086, sent: 0
Searched with beam size: 1, expanded: 600, elapsed time: 0.02376413
New primal bound: 2756, expanded: 600, elapsed time: 0.023767286
Searched with beam size: 2, threads: 10, kept: 15770, sent: 15729
Searched with beam size: 2, expanded: 1799, elapsed time: 0.052432706
New dual bound: 12, expanded: 1799, elapsed time: 0.0524346
New primal bound: 2571, expanded: 1799, elapsed time: 0.052907517
Searched with beam size: 4, threads: 10, kept: 15524, sent: 46943
Searched with beam size: 4, expanded: 4140, elapsed time: 0.086654489
Searched with beam size: 8, threads: 10, kept: 15470, sent: 108945
Searched with beam size: 8, expanded: 8884, elapsed time: 0.140594967
Searched with beam size: 16, threads: 10, kept: 25490, sent: 228463
Searched with beam size: 16, expanded: 18453, elapsed time: 0.242894231
New primal bound: 2521, expanded: 18453, elapsed time: 0.243626871
Searched with beam size: 32, t

makespan = 2493

## 離接定式化のグラフによる表示

JSP は タスクをノード, 依存関係や同時処理禁止規則をエッジで表現したグラフからエッジを選択する問題として表現することができる.

### 参考

- https://acrogenesis.com/or-tools/documentation/user_manual/manual/ls/jobshop_def_data.html
- https://zenn.dev/fusic/articles/0fed6d5dfbdeb5

### 定数

- $J$: ジョブの集合
- $M$: マシンの集合
- $O$: オペレーションの集合
- $O_j$: ジョブ $j$ のオペレーションの集合
- $O_m$: マシン $m$ で処理するオペレーションの集合
- $t_o$: オペレーション $o$ の処理時間

### グラフ

- ノード $N := O \cup \{ \text{source}, \text{target} \}$
- エッジ $E := E^c \cup E^d$
    - Conjunctive Edges $E^c$: オペレーション $o$ と $o'$ が同じジョブに属しており, $o$ の後に $o'$ を処理しなければならない場合, $(o, o') \in E^c$.
      また, $o$ があるジョブの最初のオペレーションであるとき $(\text{source}, o) \in E^c$.
      $o$ があるジョブの最後のオペレーションであるとき $(o, \text{target}) \in E^c$.
    - Disjunctive Edges $E^d$: オペレーション $o$ と $o'$ が同じマシンで処理されるとき, $(o, o') \in E^d$ かつ $(o', o) \in E^d$.
      このエッジは双方向のうちどちらかを選択し, 選択されたエッジによりオペレーションの処理順序が定まる.

このグラフのエッジで繋がれたノード(オペレーション)の間には処理順序の関係がある.
Conjunctive edge は同一ジョブ内オペレーションの順序関係を表し,
Disjunctive edge は同一マシンで処理するオペレーションの間の順序関係を表す.

### 決定変数

- $x_e \in \{ 0, 1 \} \space (e \in E)$: エッジ $e$ を選択する場合のみ $1$.
- $s_n \in \mathbb{Z} \space (n \in N)$: オペレーションの開始時刻. $\text{source}$ ノードの開始時刻は 0, 処理時間も 0 とする.

### 制約条件

- $e = (u, v)$ とする. このとき $x_e = 1 \Rightarrow s_u + t_u \leq s_v$
    - $e \in E^c \Rightarrow x_e = 1$
    - $(u, v) \in E^d \Rightarrow x_{(u,v)} + x_{(v,u)} = 1$

### 目的関数

- $s_\text{target}$ が makespan を表す. これを最小化する.

## 巡回路制約を用いた実装

上記のグラフにマシン自体をノードとして足し,
disjunctive edge のみを辿ってマシンごとに順回路を作成することでマシン内での実行順を記述することができる.

In [ ]:
class ModelCpSatArc:
    def __init__(self, jobs: list[Job]):
        machines = sorted(
            list(set(task.machine for job in jobs for task in job.tasks))
        )

        # タスクに 0 から番号を割り振る.
        # source と target はそれぞれ -1, -2 とする.
        task_indices = []
        idx = 0
        for job in jobs:
            indices = []
            for task in job.tasks:
                indices.append(idx)
                idx += 1
            task_indices.append(indices)

        all_tasks = [task for job in jobs for task in job.tasks]

        horizon = sum(task.time for job in jobs for task in job.tasks)

        model = cp_model.CpModel()

        edges = {}
        starts = {}

        # start time
        starts[-1] = model.new_constant(0)
        starts[-2] = model.new_int_var(0, horizon, "")
        for indices in task_indices:
            for idx in indices:
                starts[idx] = model.new_int_var(0, horizon, "")

        # Conjunctive Edges
        for indices in task_indices:
            edges[(-1, indices[0])] = model.new_constant(1)
            edges[(indices[-1], -2)] = model.new_constant(1)
            for i, _ in enumerate(indices):
                if i == 0:
                    continue
                edges[(indices[i - 1], indices[i])] = model.new_constant(1)

        # Disjunctive Edges
        for m in machines:
            indices_m = [
                idx
                for indices, job in zip(task_indices, jobs)
                for idx, task in zip(indices, job.tasks)
                if task.machine == m
            ] + [-3 - m]

            edges_m = {
                (u, v): model.new_bool_var("")
                for u in indices_m
                for v in indices_m
                if u != v
            }
            model.add_circuit((u, v, var) for (u, v), var in edges_m.items())

            edges |= edges_m

        for (u, v), var in edges.items():
            if u < -2 or v < -2:
                continue

            if u == -1:
                model.add(starts[u] <= starts[v])
            else:
                model.add(
                    starts[u] + all_tasks[u].time <= starts[v]
                ).only_enforce_if(edges[(u, v)])

        model.minimize(starts[-2])

        self.model = model
        self.objective = starts[-2]

    def solve(self, timeout: int = 10):
        self.solver = cp_model.CpSolver()
        self.solver.parameters.log_search_progress = True
        self.solver.parameters.max_time_in_seconds = timeout
        self.status = self.solver.solve(self.model)

In [ ]:
model1_cpsatarc = ModelCpSatArc(jobs1)
model1_cpsatarc.solve()

mo.md(f"makespan = {round(model1_cpsatarc.solver.objective_value)}")


Starting CP-SAT solver v9.15.6755
Parameters: max_time_in_seconds: 10 log_search_progress: true
Setting number of workers to 12

Initial optimization model '': (model_fingerprint: 0xaacdc7c50f505cd1)
#Variables: 291 (#ints: 1 in objective) (291 primary variables)
  - 252 Booleans in [0,1]
  - 37 in [0,197]
  - 2 constants in {0,1} 
#kCircuit: 6
#kLinear2: 222 (#enforced: 216)

Starting presolve at 0.00s
  5.31e-05s  0.00e+00d  [DetectDominanceRelations] 
  2.10e-03s  1.20e-07d  [PresolveToFixPoint] #num_loops=6 #num_dual_strengthening=1 
  2.28e-06s  0.00e+00d  [ExtractEncodingFromLinear] 
  1.19e-05s  0.00e+00d  [DetectDuplicateColumns] 
  2.11e-05s  0.00e+00d  [DetectDuplicateConstraints] 
[Symmetry] Graph for symmetry has 836 nodes and 1'403 arcs.
[Symmetry] Symmetry computation done. time: 0.000107634 dtime: 0.00010138
  1.37e-05s  0.00e+00d  [DetectDuplicateConstraintsWithDifferentEnforcements] 
  1.39e-03s  5.98e-04d  [Probe] #probed=504 #new_binary_clauses=126 
  2.58e-06s  0.0

makespan = 55

In [ ]:
model3_cpsatarc = ModelCpSatArc(jobs3)
model3_cpsatarc.solve(timeout=10)

mo.md(f"makespan = {round(model3_cpsatarc.solver.objective_value)}")


Starting CP-SAT solver v9.15.6755
Parameters: max_time_in_seconds: 10 log_search_progress: true
Setting number of workers to 12

Initial optimization model '': (model_fingerprint: 0x50689656cfb32057)
#Variables: 19'203 (#ints: 1 in objective) (19'203 primary variables)
  - 18'600 Booleans in [0,1]
  - 601 in [0,30657]
  - 2 constants in {0,1} 
#kCircuit: 20
#kLinear2: 18'030 (#enforced: 18'000)

Starting presolve at 0.00s
  3.34e-03s  0.00e+00d  [DetectDominanceRelations] 
  3.44e-01s  4.00e-07d  [PresolveToFixPoint] #num_loops=20 #num_dual_strengthening=1 
  1.05e-04s  0.00e+00d  [ExtractEncodingFromLinear] 
  5.78e-04s  0.00e+00d  [DetectDuplicateColumns] 
  5.01e-04s  0.00e+00d  [DetectDuplicateConstraints] 
[Symmetry] Graph for symmetry has 57'022 nodes and 109'799 arcs.
[Symmetry] Symmetry computation done. time: 0.00560786 dtime: 0.00701877
  6.88e-04s  0.00e+00d  [DetectDuplicateConstraintsWithDifferentEnforcements] 
  3.45e-01s  1.82e-01d  [Probe] #probed=37'200 #new_binary_cl

makespan = 24254

なんか全然ダメだった...

区間変数より circuit constraint の方がいい場合もあるらしい[^1]が, 今回はダメそう.

[^1]: https://d-krupke.github.io/cpsat-primer/04B_advanced_modelling.html